# Adding a trust layer around Responses API agent calls

This recipe shows a guardrails-as-code pattern for OpenAI agent loops that call tools. The core idea is to keep deterministic policy, schema validation, human approval, and audit tracing outside the model, then let the model focus on reasoning and language. By the end, you will have a minimal Responses API tool loop wrapped with a trust layer that can block invalid tool calls, hold sensitive actions for approval, and emit an inspectable trace. The example uses Pramagent as the concrete open-source middleware implementation, but the pattern applies to any production agent that performs side effects.

In [ ]:
%pip install -q "openai>=2.0.0" "pramagent>=0.8.3"

In [ ]:
import json
import os

from openai import OpenAI

from pramagent import Pramagent, Verdict
from pramagent.layers import ComplianceLayer, HITLLayer, ReliabilityLayer, Rule, SafetyLayer, ToolGuardLayer, ToolPolicy
from pramagent.layers.tool_guard import SideEffect
from pramagent.providers import OpenAIProvider

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

## Baseline: a small tool-using loop

The baseline loop gives the model a mock `send_email` tool and then inspects any proposed tool call. In a production service, this is the point where many agent examples immediately execute the tool. That leaves several gaps: arguments may not be schema-checked against your policy, sensitive actions may not require approval, and the decision may not produce an audit record that another engineer can inspect later.

In [ ]:
def send_email(to: str, subject: str, body: str) -> dict:
    """Mock side effect. This recipe never sends a real email."""
    return {"queued": True, "to": to, "subject": subject}

email_tool = {
    "type": "function",
    "name": "send_email",
    "description": "Queue an outbound email. This is a mock tool for the recipe.",
    "parameters": {
        "type": "object",
        "required": ["to", "subject", "body"],
        "properties": {
            "to": {"type": "string"},
            "subject": {"type": "string", "maxLength": 120},
            "body": {"type": "string", "maxLength": 2000},
        },
        "additionalProperties": False,
    },
}

baseline = client.responses.create(
    model=MODEL,
    input="Send an email to sam@example.com saying the deployment report is ready.",
    tools=[email_tool],
)

for item in baseline.output:
    if getattr(item, "type", None) == "function_call":
        print("model proposed tool:", item.name)
        print(json.dumps(json.loads(item.arguments), indent=2))

## Add a deterministic trust layer

Now put a middleware boundary between model output and tool execution. The policy below is ordinary code: it allow-lists the tool, validates arguments with JSON Schema, scopes the tool to one tenant, and requires human approval for outbound messages. This is intentionally separate from prompt instructions. If the model proposes a bad call, the policy engine decides before any side effect happens.

In [ ]:
tool_guard = ToolGuardLayer(policies=[
    ToolPolicy(
        name="send_email",
        side_effect=SideEffect.EXTERNAL_MESSAGE,
        action=Verdict.ESCALATE,
        allowed_tenants={"demo_team"},
        schema={
            "type": "object",
            "required": ["to", "subject", "body"],
            "properties": {
                "to": {"type": "string", "format": "email"},
                "subject": {"type": "string", "maxLength": 120},
                "body": {"type": "string", "maxLength": 2000},
            },
            "additionalProperties": False,
        },
        detail="outbound email requires approval",
    )
])

armor = Pramagent(
    provider=OpenAIProvider(model=MODEL, max_tokens=400, temperature=0.0),
    compliance=ComplianceLayer(),
    safety=SafetyLayer(rules=[
        Rule(
            rule_id="escalate_external_send",
            action=Verdict.ESCALATE,
            pattern=r"send\s+(an\s+)?email|notify\s+externally",
        )
    ]),
    reliability=ReliabilityLayer(timeout_s=30),
    hitl=HITLLayer(timeout_s=2),  # short timeout for the notebook; use Slack/webhook approval in production
    tool_guard=tool_guard,
    escalate_policy={"pre": "hitl", "post": "log"},
)

In [ ]:
proposed_args = {
    "to": "sam@example.com",
    "subject": "Deployment report",
    "body": "The deployment report is ready for review.",
}

decision = armor.validate_tool(
    "send_email",
    proposed_args,
    tenant_id="demo_team",
    session_id="recipe-001",
    action_label="send_email",
)
print("verdict:", decision.verdict.value)
print("reason:", decision.reason)

In [ ]:
bad_args = {
    "to": "sam@example.com",
    "subject": "Deployment report",
    "body": "The deployment report is ready.",
    "bcc_all_customers": True,
}

blocked = armor.validate_tool(
    "send_email",
    bad_args,
    tenant_id="demo_team",
    session_id="recipe-001",
    action_label="send_email",
)
print("verdict:", blocked.verdict.value)
print("reason:", blocked.reason)

wrong_tenant = armor.validate_tool(
    "send_email",
    proposed_args,
    tenant_id="contractor_team",
    session_id="recipe-001",
    action_label="send_email",
)
print("tenant verdict:", wrong_tenant.verdict.value)
print("tenant reason:", wrong_tenant.reason)

## Run the same intent through the guarded loop

The model is still called, but the side-effect path is held when policy says the action requires approval. In this notebook the approval layer uses a short timeout, so the response ends in an `idle` status instead of executing the mock email tool. In a deployed service, the same HITL boundary can be connected to Slack, a webhook, or an internal approval queue.

In [ ]:
response = await armor.run(
    "Send an email to sam@example.com saying the deployment report is ready.",
    tenant_id="demo_team",
    session_id="recipe-001",
    action="send_email",
)

print("blocked:", response.blocked)
print("hitl:", response.hitl)
print("output preview:", response.output[:240])
print("trace hash:", response.trace.this_hash)

## Inspect the trace

The trace is the part that turns a guardrail into operational evidence. Each layer records what it did, the verdict it produced, and how long it took. This makes policy enforcement inspectable after the fact without relying on a model's explanation of itself.

In [ ]:
for event in response.trace.layer_events:
    print(f"{event.layer:24} {event.decision:10} {event.detail or ''} ({event.latency_ms:.1f} ms)")

print("previous hash:", response.trace.prev_hash)
print("current hash: ", response.trace.this_hash)
print("audit chain valid:", armor.audit.verify_chain())

## When to use this pattern

This pattern is useful when an agent can trigger side effects: sending messages, writing to a database, changing configuration, moving money, or reading tenant-scoped data. It adds latency and policy surface area, so it is usually overkill for simple chat or summarization. For sensitive tool paths, the tradeoff is often worth it: the model can propose an action, but deterministic code decides whether the action is allowed, blocked, or held for approval.

## Resources

- [OpenAI Responses API documentation](https://platform.openai.com/docs/api-reference/responses)
- [OpenAI tools documentation](https://platform.openai.com/docs/guides/tools)
- [Pramagent on GitHub](https://github.com/sriram7737/pramagent)
- [Pramagent on PyPI](https://pypi.org/project/pramagent/)